In [1]:
import pandas as pd
import numpy as np
import os
import pytz

INPUT_FILE = "databento.csv"
OUTPUT_FILE = "nq_final.csv"
CHUNKSIZE = 100000

print(f"\n{'='*70}\n  STAGE 1: GENERATE CONTINUOUS CONTRACT (nq_final.csv)\n{'='*70}")

# 1. Dynamically detect time column name to prevent KeyError
sample = pd.read_csv(INPUT_FILE, nrows=5)
time_col = "datetime" if "datetime" in sample.columns else "ts_event"
print(f"  --> Detected timestamp column: '{time_col}'")


def get_cme_rollover_date(year: int, month: int) -> pd.Timestamp:
    """
    Calculates the modern CME rollover date for U.S. Equity Index Futures.
    The rollover occurs on the Monday of expiration week, which is the 
    Monday immediately preceding the third Friday of the contract month.
    """
    first_day = pd.Timestamp(year=year, month=month, day=1)
    # weekday(): 0=Mon, 1=Tue, 2=Wed, 3=Thu, 4=Fri, 5=Sat, 6=Sun
    first_friday_offset = (4 - first_day.weekday()) % 7
    first_friday = 1 + first_friday_offset
    
    # Third Friday is 1st Friday + 14 days
    third_friday_day = first_friday + 14
    third_friday = pd.Timestamp(year=year, month=month, day=third_friday_day)
    
    # Rollover Monday is the third Friday minus 4 days
    roll_monday = third_friday - pd.Timedelta(days=4)
    return roll_monday


def build_roll_schedule(start_year=2010, end_year=2030) -> dict:
    """Pre-generates modern CME index futures rollover dates (UTC localized)."""
    schedule = {}
    utc_tz = pytz.utc
    for y in range(start_year, end_year + 1):
        t_mar = get_cme_rollover_date(y, 3).tz_localize(utc_tz)
        t_jun = get_cme_rollover_date(y, 6).tz_localize(utc_tz)
        t_sep = get_cme_rollover_date(y, 9).tz_localize(utc_tz)
        t_dec = get_cme_rollover_date(y, 12).tz_localize(utc_tz)
        schedule[y] = (t_mar, t_jun, t_sep, t_dec)
    return schedule


def get_active_symbol_for_dt(dt: pd.Timestamp, roll_schedule: dict) -> str:
    """Returns the expected CME contract symbol based on modern calendar date."""
    y = dt.year
    t_mar, t_jun, t_sep, t_dec = roll_schedule[y]
    
    next_year_suffix = str((y + 1) % 10)
    curr_year_suffix = str(y % 10)
    
    if dt < t_mar:
        return f"NQH{curr_year_suffix}"
    elif dt < t_jun:
        return f"NQM{curr_year_suffix}"
    elif dt < t_sep:
        return f"NQU{curr_year_suffix}"
    elif dt < t_dec:
        return f"NQZ{curr_year_suffix}"
    else:
        return f"NQH{next_year_suffix}"


buffer = []
roll_schedule = build_roll_schedule()

# 2. Stream and clean raw Databento chunks
for i, chunk in enumerate(pd.read_csv(
    INPUT_FILE,
    usecols=[
        time_col,
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ],
    chunksize=CHUNKSIZE
)):
    print(f"  Processing chunk {i+1}...")
    
    # Standardize time column to datetime
    chunk["datetime"] = pd.to_datetime(chunk[time_col], utc=True)
    if time_col != "datetime":
        chunk.drop(columns=[time_col], inplace=True)

    # Remove spread contracts (e.g. NQH5-NQM5)
    chunk = chunk[~chunk["symbol"].str.contains("-", regex=False)]

    # Keep only NQ contracts
    chunk = chunk[chunk["symbol"].str.startswith("NQ")]
    
    # Enforce float32/int32 to save RAM
    for col in ("open", "high", "low", "close"):
        chunk[col] = chunk[col].astype(np.float32)
    chunk["volume"] = chunk["volume"].astype(np.int32)

    buffer.append(chunk)

print("  Combining chunks...")
df = pd.concat(buffer, ignore_index=True)
del buffer  # free memory
print(f"  Total raw rows loaded: {len(df):,}")

# 3. Apply modern CME Calendar Rollover Schedule
print("  Applying modern CME Calendar Rollover schedule...")
# Extract dates and map them to the active symbol on that day
df["date_temp"] = df["datetime"].dt.date
unique_dates = df["date_temp"].unique()

date_to_active = {}
for d in unique_dates:
    dt_utc = pd.Timestamp(d).tz_localize(pytz.utc)
    date_to_active[d] = get_active_symbol_for_dt(dt_utc, roll_schedule)

df["active_symbol"] = df["date_temp"].map(date_to_active)

# 4. Filter to keep ONLY the active contract for each day
continuous = df[df["symbol"] == df["active_symbol"]].copy()

# Clean and sort
continuous.sort_values("datetime", inplace=True)
continuous = continuous[["datetime", "symbol", "open", "high", "low", "close", "volume"]]
continuous.reset_index(drop=True, inplace=True)

# Save result
print(f"  Saving output to: {OUTPUT_FILE}")
continuous.to_csv(OUTPUT_FILE, index=False)

print(f"\n{'='*70}\n  STAGE 1 COMPLETE: Clean calendar-aligned contract generated!\n{'='*70}\n")
print(continuous["symbol"].value_counts())


  STAGE 1: GENERATE CONTINUOUS CONTRACT (nq_final.csv)
  --> Detected timestamp column: 'datetime'
  Processing chunk 1...
  Processing chunk 2...
  Processing chunk 3...
  Processing chunk 4...
  Processing chunk 5...
  Processing chunk 6...
  Processing chunk 7...
  Processing chunk 8...
  Processing chunk 9...
  Processing chunk 10...
  Processing chunk 11...
  Processing chunk 12...
  Processing chunk 13...
  Processing chunk 14...
  Processing chunk 15...
  Processing chunk 16...
  Processing chunk 17...
  Processing chunk 18...
  Processing chunk 19...
  Processing chunk 20...
  Processing chunk 21...
  Processing chunk 22...
  Processing chunk 23...
  Processing chunk 24...
  Processing chunk 25...
  Processing chunk 26...
  Processing chunk 27...
  Processing chunk 28...
  Processing chunk 29...
  Processing chunk 30...
  Processing chunk 31...
  Processing chunk 32...
  Processing chunk 33...
  Processing chunk 34...
  Processing chunk 35...
  Processing chunk 36...
  Process